In [7]:
import os
from torchvision import datasets
from PIL import Image
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split,Dataset
import matplotlib.pyplot as plt

ModuleNotFoundError: No module named 'matplotlib'

In [ ]:
# Path to your dataset folder (with 8 subfolders as class names)
data_path = "dataset/"

# Load dataset WITHOUT any transforms
dataset = datasets.ImageFolder(root=data_path)

# Print class names
print(f"Found {len(dataset.classes)} classes: {dataset.classes}")
print(f"Total images: {len(dataset)}")

# Access image and label
image_path, label = dataset.samples[0]
image = Image.open(image_path)

In [ ]:
from collections import Counter
from torchvision import datasets
from PIL import Image
import os

# Dataset path (with 8 subfolders as class names)
data_path = "dataset/"

# Load dataset
dataset = datasets.ImageFolder(root=data_path)

# Count occurrences of each label index
label_counts = Counter([label for _, label in dataset.samples])

# Map label index to class name
class_to_idx = dataset.class_to_idx
idx_to_class = {v: k for k, v in class_to_idx.items()}

# Print counts per class
print("Image count per class:\n")
for class_idx, count in label_counts.items():
    print(f"{idx_to_class[class_idx]}: {count} images")


In [ ]:
import os
import matplotlib.pyplot as plt
from torchvision import datasets
from PIL import Image

# Path to your dataset folder (each subfolder = one class)
data_path = "dataset/"

# Load dataset
dataset = datasets.ImageFolder(root=data_path)

# Get class-to-index mapping
class_to_idx = dataset.class_to_idx
idx_to_class = {v: k for k, v in class_to_idx.items()}

# Track which classes have been shown
shown_classes = set()

# Initialize plot
plt.figure(figsize=(15, 6))
count = 1

# Loop through dataset
for image_path, label in dataset.samples:
    if label not in shown_classes:
        # Open image
        image = Image.open(image_path)

        # Plot image
        plt.subplot(2, 4, count)  # 2 rows × 4 cols for 8 classes
        plt.imshow(image)
        plt.title(f"{idx_to_class[label]}")
        plt.axis("off")

        # Mark class as shown
        shown_classes.add(label)
        count += 1

    if len(shown_classes) == len(dataset.classes):
        break

plt.tight_layout()
plt.show()


What is Data Augmentation?
Definition: Data augmentation is the process of applying random transformations to training images to make your model more robust and generalizable.
Data augmentation is a technique used during training to artificially expand the diversity of the dataset by applying random transformations such as flipping, rotation, brightness adjustment, cropping, or scaling to input images. These augmentations are applied in real-time during data loading and help the model generalize better by exposing it to different variations of the same image. This is especially beneficial when working with small datasets, as it helps reduce overfitting and improves the model's robustness to unseen data.

In [ ]:
# Data augmentation for training
train_transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(20),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1)),
    transforms.ToTensor(),
    transforms.Normalize([0.5], [0.5])  # Grayscale; use 3 values if RGB
])

# Basic preprocessing for validation
val_transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
    transforms.Normalize([0.5], [0.5])
])


A custom dataset in PyTorch is created by subclassing torch.utils.data.Dataset and implementing the __init__, __getitem__, and __len__ methods. This approach gives full control over how the data is loaded and preprocessed, making it ideal for working with non-standard datasets such as images paired with labels in a CSV file, mixed data formats, or complex directory structures. It allows developers to define exactly how each sample (image and label) should be retrieved and prepared for the model.

In [ ]:
class CustomImageDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.root_dir = root_dir
        self.transform = transform
        self.image_paths = []
        self.labels = []
        self.class_names = sorted(os.listdir(root_dir))
        self.class_to_idx = {cls: idx for idx, cls in enumerate(self.class_names)}

        for cls in self.class_names:
            cls_folder = os.path.join(root_dir, cls)
            for img_name in os.listdir(cls_folder):
                if img_name.lower().endswith(('.png', '.jpg', '.jpeg')):
                    self.image_paths.append(os.path.join(cls_folder, img_name))
                    self.labels.append(self.class_to_idx[cls])

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        image = Image.open(self.image_paths[idx]).convert("RGB")
        label = self.labels[idx]
        if self.transform:
            image = self.transform(image)
        return image, label


In [ ]:
dataset_path = "dataset/"
full_dataset = CustomImageDataset(root_dir=dataset_path, transform=train_transform)


In [ ]:
train_size = int(0.75 * len(full_dataset))
val_size = len(full_dataset) - train_size

train_dataset, val_dataset = random_split(full_dataset, [train_size, val_size])

# Change validation set transform
val_dataset.dataset.transform = val_transform


The DataLoader in PyTorch is used to wrap a dataset and enable efficient loading of data in batches, shuffling of samples during training, and parallel loading using multiple workers. It is an essential component in training loops as it feeds the model with manageable chunks (batches) of data, allowing for faster and memory-efficient training. By abstracting the iteration over the dataset, the DataLoader simplifies and speeds up the training process significantly.



In [ ]:
batch_size = 32

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)


In [ ]:
images, labels = next(iter(train_loader))
print("Image batch shape:", images.shape)  # (batch_size, 3, 128, 128)
print("Label batch shape:", labels.shape)


In [ ]:
import matplotlib.pyplot as plt

def show_augmented_images(dataloader, class_names):
    images, labels = next(iter(dataloader))
    plt.figure(figsize=(12, 6))
    for i in range(8):
        img = images[i].permute(1, 2, 0).numpy()
        img = (img * 0.5) + 0.5  # unnormalize if using Normalize([0.5], [0.5], [0.5])
        plt.subplot(2, 4, i+1)
        plt.imshow(img)
        plt.title(class_names[labels[i]])
        plt.axis("off")
    plt.tight_layout()
    plt.show()


In [ ]:
show_augmented_images(train_loader, full_dataset.class_names)


In [ ]:
print(dataset.class_to_idx)


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.metrics import confusion_matrix, classification_report
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import os


In [ ]:
import torch.nn as nn

class DeepCNN(nn.Module):
    def __init__(self, num_classes):
        super(DeepCNN, self).__init__()
        self.net = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(64, 128, 3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Flatten(),
            nn.Dropout(0.4),
            nn.Linear(128 * 16 * 16, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        return self.net(x)


In [ ]:
import torch
import torch.optim as optim
import torch.nn as nn
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np



device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
num_classes = len(train_loader.dataset.dataset.class_names)
model = DeepCNN(num_classes).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)
best_val_acc = 0

for epoch in range(1, 21):  # You can increase/decrease epoch count
    model.train()
    total, correct, running_loss = 0, 0, 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        _, predicted = outputs.max(1)
        correct += predicted.eq(labels).sum().item()
        total += labels.size(0)

    train_acc = correct / total
    train_loss = running_loss / len(train_loader)

    # Validation
    model.eval()
    val_total, val_correct = 0, 0
    val_preds, val_targets = [], []

    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = outputs.max(1)
            val_preds.extend(predicted.cpu().numpy())
            val_targets.extend(labels.cpu().numpy())
            val_correct += predicted.eq(labels).sum().item()
            val_total += labels.size(0)

    val_acc = val_correct / val_total

    print(f"Epoch {epoch}: Train Loss = {train_loss:.4f}, Train Acc = {train_acc:.4f}, Val Acc = {val_acc:.4f}")

    # Save best model
    if val_acc > best_val_acc:
        torch.save(model.state_dict(), "best_cnn_model.pth")
        best_val_acc = val_acc


In [ ]:
# Reload best model
model.load_state_dict(torch.load("best_cnn_model.pth"))
model.eval()
class_names = dataset.classes

val_preds, val_targets = [], []
with torch.no_grad():
    for images, labels in val_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        _, predicted = outputs.max(1)
        val_preds.extend(predicted.cpu().numpy())
        val_targets.extend(labels.cpu().numpy())

# Classification Report
print("\nClassification Report:")
print(classification_report(val_targets, val_preds, target_names=class_names))

# Confusion Matrix
cm = confusion_matrix(val_targets, val_preds)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=class_names, yticklabels=class_names)
plt.title("Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.tight_layout()
plt.show()


In [ ]:
import torch
import torch.nn as nn
from torchvision import models

class CustomFineTunedDenseNet(nn.Module):
    def __init__(self, num_classes):
        super(CustomFineTunedDenseNet, self).__init__()

        # Load pretrained DenseNet121
        base_model = models.densenet121(pretrained=True)

        # Use the base DenseNet feature extractor
        self.features = base_model.features

        # Custom added layers
        self.extra_layers = nn.Sequential(
            nn.AdaptiveMaxPool2d((1, 1)),  # Global Max Pooling
            nn.Flatten(),
            nn.Linear(1024, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.extra_layers(x)
        return x


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = CustomFineTunedDenseNet(num_classes=num_classes).to(device)



criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.0005)



In [ ]:
best_val_acc = 0

for epoch in range(1, 25):
    model.train()
    running_loss, correct, total = 0, 0, 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        _, preds = torch.max(outputs, 1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    train_acc = correct / total
    train_loss = running_loss / len(train_loader)

    # Validation
    model.eval()
    val_correct, val_total = 0, 0
    val_preds, val_targets = [], []

    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs, 1)
            val_preds.extend(predicted.cpu().numpy())
            val_targets.extend(labels.cpu().numpy())
            val_correct += (predicted == labels).sum().item()
            val_total += labels.size(0)

    val_acc = val_correct / val_total

    print(f"Epoch {epoch}: Train Loss = {train_loss:.4f}, Train Acc = {train_acc:.4f}, Val Acc = {val_acc:.4f}")

    if val_acc > best_val_acc:
        torch.save(model.state_dict(), "best_densenet_model.pth")
        best_val_acc = val_acc


In [ ]:
# Ensure you use the same model architecture as during training
model.load_state_dict(torch.load("best_densenet_model.pth"))
model.eval()
val_preds = []
val_targets = []

with torch.no_grad():
    for images, labels in val_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        _, predicted = torch.max(outputs, 1)
        val_preds.extend(predicted.cpu().numpy())
        val_targets.extend(labels.cpu().numpy())
class_names = dataset.classes  # Use this for ImageFolder datasets
print("\nClassification Report:\n")
print(classification_report(val_targets, val_preds, target_names=class_names))
cm = confusion_matrix(val_targets, val_preds)

plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names)
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix")
plt.tight_layout()
plt.show()


In [ ]:
class CustomMobileNet(nn.Module):
    def __init__(self, num_classes):
        super(CustomMobileNet, self).__init__()
        base_model = models.mobilenet_v2(pretrained=True)

        #Freeze the base model (optional)
        for param in base_model.features.parameters():
            param.requires_grad = False

        self.features = base_model.features
        self.pool = nn.AdaptiveAvgPool2d(1)  # Global average pooling

        # Custom classifier
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(1280, 512),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(512, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.pool(x)
        x = self.classifier(x)
        return x


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
num_classes = len(dataset.classes)

model = CustomMobileNet(num_classes=num_classes).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.0005)


In [ ]:
best_val_acc = 0

for epoch in range(1, 21):
    model.train()
    running_loss, correct, total = 0, 0, 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        _, preds = outputs.max(1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    train_acc = correct / total
    train_loss = running_loss / len(train_loader)

    # Validation
    model.eval()
    val_correct, val_total = 0, 0
    val_preds, val_targets = [], []

    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs, 1)
            val_preds.extend(predicted.cpu().numpy())
            val_targets.extend(labels.cpu().numpy())
            val_correct += (predicted == labels).sum().item()
            val_total += labels.size(0)

    val_acc = val_correct / val_total

    print(f"Epoch {epoch}: Train Loss = {train_loss:.4f}, Train Acc = {train_acc:.4f}, Val Acc = {val_acc:.4f}")

    if val_acc > best_val_acc:
        torch.save(model.state_dict(), "best_mobilenet_model.pth")
        best_val_acc = val_acc


In [ ]:
# Ensure you use the same model architecture as during training
model.load_state_dict(torch.load("best_mobilenet_model.pth"))
model.eval()
val_preds = []
val_targets = []

with torch.no_grad():
    for images, labels in val_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        _, predicted = torch.max(outputs, 1)
        val_preds.extend(predicted.cpu().numpy())
        val_targets.extend(labels.cpu().numpy())
class_names = dataset.classes  # Use this for ImageFolder datasets
print("\nClassification Report:\n")
print(classification_report(val_targets, val_preds, target_names=class_names))
cm = confusion_matrix(val_targets, val_preds)

plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names)
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix")
plt.tight_layout()
plt.show()


In [ ]:
# Device setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Paths
dataset_path = "dataset/"
batch_size = 32
num_epochs = 5

# Transform (same used during training & feature extraction)
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

# Load dataset
dataset = datasets.ImageFolder(dataset_path, transform=transform)
num_classes = len(dataset.classes)

train_loader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

# Load pretrained ResNet
model = models.resnet18(pretrained=True)
model.fc = nn.Linear(model.fc.in_features, num_classes)
model = model.to(device)

# Loss and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.0005)

# Training loop
for epoch in range(1, num_epochs + 1):
    model.train()
    running_loss, correct, total = 0, 0, 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        _, predicted = torch.max(outputs, 1)
        correct += (predicted == labels).sum().item()
        total += labels.size(0)

    acc = correct / total
    print(f"Epoch {epoch}/{num_epochs} - Loss: {running_loss:.4f} - Accuracy: {acc:.4f}")

# Save trained model
torch.save(model.state_dict(), "resnet18_trained.pth")



In [ ]:

from tqdm import tqdm
#Reload model
resnet_feat = models.resnet18(pretrained=False)
resnet_feat.fc = nn.Linear(resnet_feat.fc.in_features, num_classes)
resnet_feat.load_state_dict(torch.load("resnet18_trained.pth"))

# Remove classifier
resnet_feat.fc = nn.Identity()
resnet_feat = resnet_feat.to(device)
resnet_feat.eval()

# Use same dataset but in order
loader = DataLoader(dataset, batch_size=batch_size, shuffle=False)

features, labels = [], []

with torch.no_grad():
    for images, lbls in tqdm(loader, desc="Extracting features"):
        images = images.to(device)
        outputs = resnet_feat(images)  # Shape: [batch, 512]
        features.extend(outputs.cpu().numpy())
        labels.extend(lbls.numpy())

features = np.array(features)
labels = np.array(labels)


In [ ]:
features.shape

In [ ]:
labels.shape

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
# Train-test split
X_train, X_test, y_train, y_test = train_test_split(features, labels, test_size=0.2, random_state=42)

# Train RF
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)

# Predict and evaluate
y_pred = rf.predict(X_test)

print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=dataset.classes))




In [ ]:
# y_test and y_pred should already be available
cm = confusion_matrix(y_test, y_pred)
class_names = dataset.classes  # From your ImageFolder
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names,
            yticklabels=class_names)
plt.title('Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.tight_layout()
plt.show()


In [ ]:
import joblib

# Save model
joblib.dump(rf, "random_forest_model.pkl")
print("Random Forest model saved as random_forest_model.pkl")


In [ ]:
import torch
from torchvision import models, transforms
from PIL import Image
import joblib
import numpy as np
import matplotlib.pyplot as plt

# Setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
image_path = "path_to_your_image.jpg"  # 🖼️ Replace with your actual image path
resnet_model_path = "resnet18_trained.pth"
rf_model_path = "random_forest_model.pkl"
class_names = ["class_0", "class_1", "class_2"]  # 🔄 Replace with actual classes used

# Transform
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

# Load and preprocess image
image = Image.open(image_path).convert("RGB")
input_tensor = transform(image).unsqueeze(0).to(device)

# Load ResNet18 for feature extraction
resnet = models.resnet18(pretrained=False)
resnet.fc = torch.nn.Linear(resnet.fc.in_features, len(class_names))
resnet.load_state_dict(torch.load(resnet_model_path))
resnet.fc = torch.nn.Identity()
resnet = resnet.to(device)
resnet.eval()

# Extract features
with torch.no_grad():
    features = resnet(input_tensor).cpu().numpy()  # Shape: [1, 512]

# Load Random Forest model
rf_model = joblib.load(rf_model_path)
predicted_index = rf_model.predict(features)[0]
predicted_label = class_names[predicted_index]

# Display image and predicted class
plt.imshow(image)
plt.title(f"Predicted: {predicted_label}", fontsize=16)
plt.axis('off')
plt.show()


In [ ]:
import torch
from torchvision import models, transforms
from PIL import Image
import joblib
import numpy as np
import matplotlib.pyplot as plt

# Paths
image_path = r"Z:\Arjun k p\Bacteria detection\Dataset\Yeast\Yeast_10.png"  
resnet_model_path = "resnet18_trained.pth"
rf_model_path = "random_forest_model.pkl"

# Device setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Class labels
class_names = [
    'Amoeba', 'Euglena', 'Hydra', 'Paramecium',
    'Rod Bacteria', 'Spherical Bacteria',
    'Spiral Bacteria', 'Yeast'
]

# Transform
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

# Load and preprocess image
image = Image.open(image_path).convert("RGB")
input_tensor = transform(image).unsqueeze(0).to(device)

# Load ResNet18 for feature extraction
resnet = models.resnet18(pretrained=False)
resnet.fc = torch.nn.Linear(resnet.fc.in_features, len(class_names))
resnet.load_state_dict(torch.load(resnet_model_path))
resnet.fc = torch.nn.Identity()  # Remove final classifier
resnet = resnet.to(device)
resnet.eval()

# Extract features
with torch.no_grad():
    features = resnet(input_tensor).cpu().numpy()

# Load Random Forest and predict
rf_model = joblib.load(rf_model_path)
predicted_index = rf_model.predict(features)[0]
predicted_label = class_names[predicted_index]

# Show image with prediction
plt.imshow(image)
plt.title(f"Predicted Class: {predicted_label}", fontsize=16)
plt.axis('off')
plt.show()


In [ ]:

import torch
from torchvision import models, transforms
from PIL import Image
import joblib
import numpy as np
import matplotlib.pyplot as plt

# Paths
image_path = r"Z:\Arjun k p\Bacteria detection\Dataset\Amoeba\Amoeba_38.png" 
resnet_model_path = "resnet18_trained.pth"
rf_model_path = "random_forest_model.pkl"

# Device setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Class labels
class_names = [
    'Amoeba', 'Euglena', 'Hydra', 'Paramecium',
    'Rod Bacteria', 'Spherical Bacteria',
    'Spiral Bacteria', 'Yeast'
]

# Transform
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

# Load and preprocess image
image = Image.open(image_path).convert("RGB")
input_tensor = transform(image).unsqueeze(0).to(device)

# Load ResNet18 for feature extraction
resnet = models.resnet18(pretrained=False)
resnet.fc = torch.nn.Linear(resnet.fc.in_features, len(class_names))
resnet.load_state_dict(torch.load(resnet_model_path))
resnet.fc = torch.nn.Identity()  # Remove final classifier
resnet = resnet.to(device)
resnet.eval()

# Extract features
with torch.no_grad():
    features = resnet(input_tensor).cpu().numpy()

# Load Random Forest and predict
rf_model = joblib.load(rf_model_path)
predicted_index = rf_model.predict(features)[0]
predicted_label = class_names[predicted_index]

# Show image with prediction
plt.imshow(image)
plt.title(f"Predicted Class: {predicted_label}", fontsize=16)
plt.axis('off')
plt.show()


In [ ]:

import torch
from torchvision import models, transforms
from PIL import Image
import joblib
import numpy as np
import matplotlib.pyplot as plt

# Paths
image_path = r"Z:\Arjun k p\Bacteria detection\Dataset\Hydra\Hydra_11.png" 
resnet_model_path = "resnet18_trained.pth"
rf_model_path = "random_forest_model.pkl"

# Device setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Class labels
class_names = [
    'Amoeba', 'Euglena', 'Hydra', 'Paramecium',
    'Rod Bacteria', 'Spherical Bacteria',
    'Spiral Bacteria', 'Yeast'
]

# Transform
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

# Load and preprocess image
image = Image.open(image_path).convert("RGB")
input_tensor = transform(image).unsqueeze(0).to(device)

# Load ResNet18 for feature extraction
resnet = models.resnet18(pretrained=False)
resnet.fc = torch.nn.Linear(resnet.fc.in_features, len(class_names))
resnet.load_state_dict(torch.load(resnet_model_path))
resnet.fc = torch.nn.Identity()  # Remove final classifier
resnet = resnet.to(device)
resnet.eval()

# Extract features
with torch.no_grad():
    features = resnet(input_tensor).cpu().numpy()

# Load Random Forest and predict
rf_model = joblib.load(rf_model_path)
predicted_index = rf_model.predict(features)[0]
predicted_label = class_names[predicted_index]

# Show image with prediction
plt.imshow(image)
plt.title(f"Predicted Class: {predicted_label}", fontsize=16)
plt.axis('off')
plt.show()
